# 10강 — TensorFlow Lite 모델 변환 및 Edge 추론

TensorFlow/Keras Model → 모델 학습 → TFLite 변환 → `.tflite` 저장 →
Interpreter 로딩 → Sensor 데이터 입력 → Inference → NORMAL/ANOMALY 판단

이 노트북에서 만든 `motor_anomaly_model.tflite`는 다음 폴더
(`../10-tflite-deploy`)에서 **EdgeX Go App Service가 실제로 로드해서 추론**합니다.
학습은 EdgeX와 무관한 이 가벼운 환경에서, 배포는 EdgeX 위에서 — 라는 흐름입니다.

In [ ]:
import numpy as np
import tensorflow as tf

## 1. 학습 데이터 생성 (입력: [온도, 진동])

In [ ]:
X = np.array([
    [60, 2.0], [65, 2.5], [70, 3.0], [72, 3.2], [75, 3.5], [78, 4.0],
    [82, 5.0], [85, 5.5], [90, 7.0], [92, 7.5], [95, 8.0], [98, 8.5]
], dtype=np.float32)

# 0 = NORMAL, 1 = ANOMALY
y = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], dtype=np.float32)

## 2. 입력 데이터 정규화 (교육용 단순 Scaling)

In [ ]:
X_scaled = X.copy()
X_scaled[:, 0] = X_scaled[:, 0] / 100.0
X_scaled[:, 1] = X_scaled[:, 1] / 10.0

## 3~4. TensorFlow/Keras 모델 생성 및 학습

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(X_scaled, y, epochs=150, verbose=0)
print("TensorFlow Model Training Complete")

## 5~6. TensorFlow Lite 변환 및 저장

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

MODEL_PATH = "motor_anomaly_model.tflite"
with open(MODEL_PATH, "wb") as f:
    f.write(tflite_model)

print("TFLite Model Saved :", MODEL_PATH, f"({len(tflite_model)} bytes)")

## 7~8. Interpreter 로딩 및 Tensor 정보 확인

In [ ]:
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input Details\n", input_details)
print("\nOutput Details\n", output_details)

## 9~12. Edge Sensor 데이터로 추론

In [ ]:
temperature = 91.0
vibration = 7.3

sensor_input = np.array([[temperature / 100.0, vibration / 10.0]], dtype=np.float32)

interpreter.set_tensor(input_details[0]["index"], sensor_input)
interpreter.invoke()

prediction = interpreter.get_tensor(output_details[0]["index"])
anomaly_score = float(prediction[0][0])

print("Anomaly Score :", round(anomaly_score, 4))

## 13~14. Postprocessing 및 최종 결과

In [ ]:
THRESHOLD = 0.5
status = "ANOMALY" if anomaly_score >= THRESHOLD else "NORMAL"

print("\n========================")
print("Edge AI Result")
print("========================")
print("Temperature :", temperature)
print("Vibration   :", vibration)
print("AI Score    :", round(anomaly_score, 4))
print("Status      :", status)

if status == "ANOMALY":
    print(">>> ALERT : 설비 이상 가능성 감지")

## 다음 단계

이 셀들이 만든 `motor_anomaly_model.tflite`를 다운로드해서
`../10-tflite-deploy/app/model/motor_anomaly_model.tflite`로 옮기면
(이미 검증된 모델이 기본으로 들어있습니다) EdgeX Go App Service가
Motor01의 실시간 센서 값으로 **바로 이 모델**을 사용해 추론합니다.